<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module12/Lab3.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 3 — Building the QAOA Cost Operator (Instructor)
**Quantum Optimization and Simulation — QAOA Laboratory Series**

Instructor version with completed exercises and answer key.

**Format:** 10–15 minute instructor walkthrough + about 45–60 minutes of independent work.

**Notebook style:** Most code is supplied. Cells marked **YOUR TURN** contain a small value, line, or function for you to complete.

> Qiskit displays measured bitstrings in the order `q_(n-1)...q_0`. When we discuss graph nodes, this notebook often converts them to `q_0...q_(n-1)` using `q0_first(...)`.

## Learning goals
- Understand \(ZZ\) parity.
- Build \(R_{ZZ}\) using `CX–RZ–CX`.
- See how a Max-Cut edge can be encoded as a parity-dependent phase.

In [ ]:
# Run this once at the beginning of a fresh Google Colab session.
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-optimization~=0.7" "qiskit-ibm-runtime~=0.46"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_aer.primitives import SamplerV2

SEED = 123
SHOTS = 2048
sampler = SamplerV2(default_shots=SHOTS, seed=SEED)

def run_counts(qc, shots=SHOTS):
    "Run a measured circuit with Aer SamplerV2 and return counts."
    result = sampler.run([qc], shots=shots).result()
    return result[0].data.meas.get_counts()

def q0_first(qiskit_bitstring):
    "Convert Qiskit's displayed q_(n-1)...q_0 bitstring to q_0...q_(n-1)."
    return qiskit_bitstring.replace(" ", "")[::-1]

## Part A — One edge

For an edge \((i,j)\), Max-Cut rewards states where the two endpoint bits are different.

In the lecture convention, after dropping the identity/global-phase term, one edge contributes a cost Hamiltonian proportional to

\[
-\frac{1}{2} Z_i Z_j.
\]

Therefore its QAOA cost unitary can be implemented with `CX – RZ(-gamma) – CX`.

In [ ]:
def apply_edge_cost(qc, i, j, gamma):
    qc.cx(i, j)
    qc.rz(-gamma, j)
    qc.cx(i, j)

## Part B — Verify all four computational basis states with a statevector

In [ ]:
from qiskit.quantum_info import Statevector

gamma = np.pi / 3

for bits_q0first in ["00", "01", "10", "11"]:
    qc = QuantumCircuit(2)

    for q, b in enumerate(bits_q0first):
        if b == "1":
            qc.x(q)

    apply_edge_cost(qc, 0, 1, gamma)
    sv = Statevector.from_instruction(qc)

    amps = sv.to_dict()
    nonzero = {k:v for k,v in amps.items() if abs(v) > 1e-10}
    print(bits_q0first, nonzero)

**Expected:** `00` and `11` acquire one phase sign; `01` and `10` acquire the opposite sign. Their probabilities remain 1 because this operation is phase-only on computational basis states.

## Part C — YOUR TURN: build a 3-qubit, 2-edge cost layer

In [ ]:
edges = [(0, 1), (1, 2)]
gamma = 0.7

qc = QuantumCircuit(3)
qc.h(range(3))

for i, j in edges:
    # TODO: replace pass with the three-gate edge-cost sequence.
    pass

display(qc.draw("mpl"))

**Expected circuit:** H on all three qubits, plus one `CX–RZ(-gamma)–CX` block for each edge.

## Questions
1. Why does the first CNOT help us use a single-qubit \(R_Z\) to create a two-qubit parity-dependent phase?
2. Why does the second CNOT appear?
3. Does this cost layer by itself favor a bitstring in the Z-basis histogram? Explain.

## Instructor solutions

In [ ]:
edges = [(0, 1), (1, 2)]
gamma = 0.7

qc = QuantumCircuit(3)
qc.h(range(3))

for i, j in edges:
    qc.cx(i, j)
    qc.rz(-gamma, j)
    qc.cx(i, j)

display(qc.draw("mpl"))

1. The first CNOT temporarily writes the XOR/parity of the pair onto the target.
2. The second CNOT uncomputes that temporary parity, restoring the computational bits while retaining the phase.
3. No. The cost layer changes phases but not the magnitudes of computational-basis amplitudes. A mixer is needed to convert phase differences into probability differences.